In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import os
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

Q1_data_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(Q1_data_path)


In [ ]:
# Task 2: Write your code here:
df.head()



In [ ]:

# Task 3: Write your code here:
df.info()


In [ ]:
# Task 4: Write your code here:
df.describe()


In [ ]:
# Task 5: Write your code here:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

print("\nCleaned column names:")
print(df.columns.tolist())
plt.figure()
df['delivery_time'].hist()
plt.xlabel("Delivery Time")
plt.ylabel("Frequency")
plt.title("Target Distribution: Delivery Time")
plt.show()



In [ ]:
# Task 1: Write your code here:
if 'order_id' in df.columns:
    df.drop(columns=['order_id'], inplace=True)

df.head()


In [ ]:
# Task 2: Write your code here:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Fill numerical columns with median
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill categorical columns with mode
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)


In [ ]:
df.isnull().sum()


In [ ]:
# Task 3: Write your code here:
# Separate numerical and categorical columns
df.duplicated().sum()
# Remove duplicates if any
df.drop_duplicates(inplace=True)

# Confirm removal
df.duplicated().sum()




In [ ]:
# Task 4: Write your code here:
df = pd.get_dummies(df, drop_first=True)

df.head()



In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df.drop(columns=['delivery_time'])
y = df['delivery_time']

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)



In [ ]:
# Task 6: Write your code here:



In [ ]:
# Task 1: Write your code here:
# Features and target
X = X_scaled              # scaled features from Part 2
y = df['delivery_time'].values


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# K-Fold Cross Validation (correct for regression)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Random Forest model
    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # MAE evaluation
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

# Averaged MAE across all folds
print(f"Average MAE across all folds: {np.mean(mae_scores):.4f}")


In [ ]:
# Task 1: Write your code here:
from sklearn.ensemble import RandomForestRegressor

final_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

final_model.fit(X, y)

import matplotlib.pyplot as plt
import pandas as pd

importance_df = pd.DataFrame({
    'feature': df.drop(columns=['delivery_time']).columns,
    'importance': final_model.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure()
plt.barh(importance_df['feature'], importance_df['importance'])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Feature Importance")
plt.show()


In [ ]:
# Task 2: Write your code here:
y_pred = final_model.predict(X)

plt.figure()
plt.hist(y_pred, bins=30)
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Frequency")
plt.title("Predicted Delivery Time Distribution")
plt.show()


In [ ]:
# Task Bonus: Write your code here:
!pip install catboost

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
import numpy as np

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Model 1: Random Forest
    rf_model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    # Model 2: CatBoost (silent mode)
    cb_model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        random_state=42,
        verbose=False
    )

    # Train both models
    rf_model.fit(X_train, y_train)
    cb_model.fit(X_train, y_train)

    # Predictions
    rf_pred = rf_model.predict(X_test)
    cb_pred = cb_model.predict(X_test)

    # Average predictions (ensemble)
    avg_pred = (rf_pred + cb_pred) / 2

    # MAE on averaged predictions
    mae = mean_absolute_error(y_test, avg_pred)
    mae_scores.append(mae)

# Average MAE across folds
print(f"Average MAE (Ensemble RF + CatBoost): {np.mean(mae_scores):.4f}")
